<a href="https://colab.research.google.com/github/Quantx73/pairs-trading-ml-v2/blob/main/notebooks/Pairs_Trading_ML_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pairs Trading Strategy Enhanced with Machine Learning
## A Practical Implementation on ETF Pairs

**Author:** Nabyl H.  
**Date:** March 2026  
**Course:** Certificate in Python for Finance (CPF)

In [1]:
# Cell 1: Load static CSV data

import pandas as pd
from pathlib import Path
import urllib.request

# Create data folder if needed
Path("data").mkdir(exist_ok=True)
csv_path = Path("data") / "etf_pairs.csv"

# Download if missing (from same GitHub repo, not external API)
if not csv_path.exists():
    url = "https://raw.githubusercontent.com/Quantx73/pairs-trading-ml-v2/main/data/etf_pairs.csv"
    urllib.request.urlretrieve(url, csv_path)
    print("CSV downloaded from GitHub")

# Load data
prices = pd.read_csv(csv_path, index_col=0, parse_dates=True)

print("Data loaded from", csv_path)
print("Shape:", prices.shape)
print(prices.head())

CSV downloaded from GitHub
Data loaded from data/etf_pairs.csv
Shape: (2515, 2)
                  EWA        EWC
Date                            
2015-01-02  13.892488  22.766460
2015-01-05  13.760474  22.155445
2015-01-06  13.703896  21.830095
2015-01-07  13.829624  21.853901
2015-01-08  14.011926  22.123701


In [ ]:
# Cell 2: Plot EWA and EWC prices

import matplotlib.pyplot as plt

# Simple line plot
plt.figure(figsize=(12, 5))
plt.plot(prices.index, prices['EWA'], label='EWA (Australia)', linewidth=1)
plt.plot(prices.index, prices['EWC'], label='EWC (Canada)', linewidth=1)
plt.title('ETF Prices: EWA vs EWC')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 3: Baseline z-score strategy

# Parameters
window = 60
entry = 2.0
exit_threshold = 0.5
cost = 0.001

# Price ratio and z-score
prices['ratio'] = prices['EWA'] / prices['EWC']
prices['ratio_ma'] = prices['ratio'].rolling(window).mean()
prices['ratio_std'] = prices['ratio'].rolling(window).std()
prices['zscore'] = (prices['ratio'] - prices['ratio_ma']) / prices['ratio_std']

# Entry signals
prices['signal'] = 0
prices.loc[prices['zscore'] < -entry, 'signal'] = 1   # long spread
prices.loc[prices['zscore'] >  entry, 'signal'] = -1  # short spread

# Simulate positions (simple state machine)
position = 0
pos_list = []
for i in range(len(prices)):
    sig = prices['signal'].iloc[i]
    if position == 0:
        if sig == 1:
            position = 1
        elif sig == -1:
            position = -1
    elif position == 1:
        if abs(prices['zscore'].iloc[i]) < exit_threshold or sig == -1:
            position = 0
    elif position == -1:
        if abs(prices['zscore'].iloc[i]) < exit_threshold or sig == 1:
            position = 0
    pos_list.append(position)
prices['position'] = pos_list

# Returns
prices['ret_A'] = prices['EWA'].pct_change()
prices['ret_B'] = prices['EWC'].pct_change()
prices['ret_spread'] = prices['ret_A'] - prices['ret_B']
prices['strat_ret'] = prices['position'].shift(1) * prices['ret_spread']

# Transaction costs (each change of position)
prices['trade'] = prices['position'].diff().abs() / 2
prices['strat_ret_net'] = prices['strat_ret'] - prices['trade'] * cost

# Cumulative return
prices['cum_ret'] = (1 + prices['strat_ret_net']).cumprod()

# Summary
total_return = prices['cum_ret'].iloc[-1] - 1
print(f"Baseline total return (net): {total_return:.2%}")


In [ ]:
# Cell 4: Plot baseline results

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Top: z-score with entry/exit lines
ax1.plot(prices.index, prices['zscore'], color='blue', linewidth=1)
ax1.axhline(y=entry, color='red', linestyle='--', label='Entry +2')
ax1.axhline(y=-entry, color='red', linestyle='--', label='Entry -2')
ax1.axhline(y=exit_threshold, color='orange', linestyle=':', label='Exit ±0.5')
ax1.axhline(y=-exit_threshold, color='orange', linestyle=':')
ax1.set_title('Z-Score and Trading Signals')
ax1.set_ylabel('Z-Score')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Bottom: cumulative returns
ax2.plot(prices.index, prices['cum_ret'], color='green', linewidth=1)
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Cumulative Returns (net of transaction costs)')
ax2.set_xlabel('Date')
ax2.set_ylabel('Cumulative Return')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Feature engineering (no leakage)

window = 60
lags = [1, 2, 3, 5]

# Volatility
prices['vol_A'] = prices['ret_A'].rolling(window).std() * (252**0.5)
prices['vol_B'] = prices['ret_B'].rolling(window).std() * (252**0.5)
prices['vol_ratio'] = prices['vol_A'] / prices['vol_B']

# Correlation
prices['corr'] = prices['ret_A'].rolling(window).corr(prices['ret_B'])

# Lagged z-scores
for lag in lags:
    prices[f'zscore_lag_{lag}'] = prices['zscore'].shift(lag)

# Lagged returns
for lag in lags:
    prices[f'ret_A_lag_{lag}'] = prices['ret_A'].shift(lag)
    prices[f'ret_B_lag_{lag}'] = prices['ret_B'].shift(lag)

# Target
horizon = 5
prices['future_zscore'] = prices['zscore'].shift(-horizon)
prices['target'] = ((abs(prices['future_zscore']) < abs(prices['zscore'])) &
                    (abs(prices['zscore']) > 1.5)).astype(int)

# Drop NaNs
features = prices.dropna()

# Exclude all columns that would cause leakage
exclude = ['EWA', 'EWC', 'target', 'future_zscore',
           'ratio', 'ratio_ma', 'ratio_std', 'signal', 'position',
           'ret_spread', 'strat_ret', 'strat_ret_net', 'trade', 'cum_ret',
           'return_A', 'return_B', 'ret_A', 'ret_B']

feature_cols = [c for c in features.columns if c not in exclude]

print("Features:", len(feature_cols))
print("Samples:", len(features))


In [ ]:
# Cell 6 : Quick checks
print("Nombre de trades:", (prices['position'].diff() != 0).sum() // 2)
print("Dernier z-score:", prices['zscore'].iloc[-1])
print("Dernière position:", prices['position'].iloc[-1])
print("Premier jour avec position non nulle:")
print(prices[prices['position'] != 0].head())

In [ ]:
# Cell 7: Random Forest model

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X = features[feature_cols].values
y = features['target'].values

split = int(len(X) * 0.8)
X_train = X[:split]
X_test = X[split:]
y_train = y[:split]
y_test = y[split:]

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['No trade', 'Trade']))

imp = rf.feature_importances_
top = imp.argsort()[-5:][::-1]
print("\nTop 5 features:")
for i in top:
    print(" ", feature_cols[i], round(imp[i], 4))


In [ ]:
# Cell 8: Plot feature importances

import matplotlib.pyplot as plt

n = 10
idx = imp.argsort()[-n:][::-1]
names = [feature_cols[i] for i in idx]
vals = imp[idx]

plt.figure(figsize=(8,5))
plt.barh(range(n), vals[::-1], color='steelblue')
plt.yticks(range(n), names[::-1])
plt.xlabel('Importance')
plt.title(f'Top {n} Features')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9: Print top 5 features

print("Top 5 features:")
for i in range(5):
    print(f"  {i+1}. {feature_cols[top[i]]}: {imp[top[i]]:.4f}")

In [ ]:
# Cell 10: ML backtest on test set

cost = 0.001

test = features.iloc[split:].copy()
Xtest = test[feature_cols].values
preds = rf.predict(Xtest)

test['ml_sig'] = 0
test.loc[(test['zscore'] < -1.5) & (preds == 1), 'ml_sig'] = 1
test.loc[(test['zscore'] > 1.5) & (preds == 1), 'ml_sig'] = -1

pos = 0
plist = []
for s in test['ml_sig']:
    if pos == 0:
        if s == 1:
            pos = 1
        elif s == -1:
            pos = -1
    elif pos == 1 and s == -1:
        pos = 0
    elif pos == -1 and s == 1:
        pos = 0
    plist.append(pos)
test['ml_pos'] = plist

test['ml_ret'] = test['ml_pos'].shift(1) * test['ret_spread']
test['ml_trade'] = test['ml_pos'].diff().abs() / 2
test['ml_ret_net'] = test['ml_ret'] - test['ml_trade'] * cost
test['ml_cum'] = (1 + test['ml_ret_net']).cumprod()

print("ML total return (net):", f"{test['ml_cum'].iloc[-1] - 1:.2%}")
print("ML trades:", test['ml_trade'].sum())

In [ ]:
# Cell 11: Compare baseline and ML on test period

# Baseline on same dates
base_test = prices.loc[test.index].copy()
base_cum = base_test['cum_ret']
base_ret = base_test['strat_ret_net']

# ML from cell 10
ml_cum = test['ml_cum']
ml_ret = test['ml_ret_net']

# Helper
def stats(cum, ret):
    tot = cum.iloc[-1] - 1
    ann = (1 + tot) ** (252/len(cum)) - 1
    vol = ret.std() * (252**0.5)
    sharpe = ann / vol if vol != 0 else 0
    dd = (cum / cum.cummax() - 1).min()
    wr = (ret > 0).mean()
    return tot, ann, sharpe, dd, wr

tot_b, ann_b, sharpe_b, dd_b, wr_b = stats(base_cum, base_ret)
tot_m, ann_m, sharpe_m, dd_m, wr_m = stats(ml_cum, ml_ret)

print("="*50)
print("BASELINE vs ML (test period only)")
print("="*50)
print(f"Metric               Baseline     ML")
print("-"*50)
print(f"Total return         {tot_b:>10.2%}   {tot_m:>10.2%}")
print(f"Annual return        {ann_b:>10.2%}   {ann_m:>10.2%}")
print(f"Sharpe ratio         {sharpe_b:>10.2f}   {sharpe_m:>10.2f}")
print(f"Max drawdown         {dd_b:>10.2%}   {dd_m:>10.2%}")
print(f"Win rate             {wr_b:>10.2%}   {wr_m:>10.2%}")
print(f"Number of trades     {base_test['trade'].sum():>10.0f}   {test['ml_trade'].sum():>10.0f}")

# 7. Conclusion

## Summary
I compared a classic pairs trading strategy (z-score) with a Random Forest model on EWA/EWC ETFs from 2015 to 2024. The test period is the last 20% of the data (about 2019-2024).

## Main results
- The baseline z-score (window 60, entry ±2, exit 0.5) returned **68.30% net** on the test period, with a Sharpe ratio of 5.06.
- The Random Forest returned **15.89% net** with a Sharpe of 0.82.
- ML has a better win rate (49.4% vs 16.7%) but smaller gains per trade.

## Why ML did not beat the baseline?
The EWA/EWC pair was strongly cointegrated during the test period. The z-score alone captured most opportunities. ML avoided some false signals (higher win rate) but also missed some very profitable trades.

## Limitations
- No walk-forward backtest (simple train/test split).
- Only one pair tested.
- Fixed transaction costs (0.1%), no slippage.

## Future work
- Test on multiple pairs (sectors, indices).
- Use LSTM or Transformer for longer time dependencies.
- Implement rolling retraining (walk-forward).
- Add macro or sentiment features.

## Code and data
The full notebook is on GitHub: [Quantx73/pairs-trading-ml-v2](https://github.com/Quantx73/pairs-trading-ml-v2).  
Data is static (CSV) and everything runs on Google Colab .


In [ ]:
# Cell 13: LSTM model

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

print("1. Imports OK")

seq_len = 10

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features[feature_cols].values)
print("2. Scaling done, shape:", X_scaled.shape)

# Build sequences
X, y = [], []
for i in range(seq_len, len(X_scaled)):
    X.append(X_scaled[i-seq_len:i])
    y.append(features['target'].values[i])
X = np.array(X)
y = np.array(y)
print("3. Sequences built, X shape:", X.shape)

# Split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print("4. Split done, train shape:", X_train.shape)

# Model
model = Sequential()
model.add(LSTM(64, return_sequences=True, input_shape=(seq_len, X.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(32))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("5. Model compiled")

# Train
early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, callbacks=[early], verbose=1)
print("6. Training done")

# Eval
pred = (model.predict(X_test) > 0.5).astype(int)
acc = (pred == y_test).mean()
print(f"LSTM test accuracy: {acc:.4f}")